# App 8 · 企业知识助手

员工问「入职 7 年有几天年假」，助手要查制度；问订单 ORD-002 的状态，则要查订单系统。两个问题从同一个入口进来。

```
                    User Question
                          ↓
              ┌──────────────────────┐
              │   PlannerAgent       │  ← Multi-Agent
              │   决定走哪条路径      │
              └──┬─────────┬───────┬─┘
                 │         │       │
       ┌─────────┘         │       └────────┐
       ▼                   ▼                ▼
┌──────────────┐  ┌─────────────────┐  ┌────────────┐
│ Agentic RAG  │  │ MCP Tool Call   │  │ Direct LLM │  ← MCP, RAG
│ (向量检索)│  │ (订单/库存)      │  │            │
└──────┬───────┘  └─────────┬───────┘  └─────┬──────┘
       │                    │                │
       └─────────┐    ┌─────┘    ┌───────────┘
                 ▼    ▼    ▼
           ┌──────────────────────┐
           │ ReviewerAgent        │  ← Multi-Agent
           └─────────┬────────────┘
                     ▼
           ┌──────────────────┐
           │ Final Answer     │
           └──────────────────┘

整条 pipeline wrapped in @observe → trace tree    ← LLMOps
最终打包成 Skill 给团队复用                        ← Skills
```

完整可跑代码在 [`Applications/skills_demo/capstone_assistant/`](../Applications/skills_demo/capstone_assistant/) ——它是一个真实的 Anthropic Skill 文件夹，含 `SKILL.md` + `pipeline.py` + `eval.py` + `reference/`。**任何团队拿到这个文件夹就能 fork 改造**。这就是 Skill 格式相对于"散落代码 + 文档 + Wiki"的核心价值：**工程整合 = 一个可复制的目录**。

下面我们通过统一入口加载这个 Skill，跑 4 类 query 看路由，最后用打包的 batch eval 评测综合表现。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"[DIR] repo root: {_root}")


[DIR] repo root: .


In [2]:
from utils.config import setup
env = setup()
from utils.multi_agent import Message, MessageType, BaseAgent, Orchestrator
from utils.mcp_helpers import EduMCPServer, EduMCPClient, ToolDef, tool_from_function
from utils.embedding_backend import SimpleVectorStore
from utils.observability import observer, observe, span, get_backend
import json, time, sys
from pathlib import Path

# 引入 Day5 上午写的 Agentic RAG 工具 (重新建 vector store + bm25)
sys.path.insert(0, str(Path('Applications/mcp_server_demo')))

llm = env.get_llm()
embedder = env.get_embedder()

print(f"OK Observability backend: {get_backend()}  (mock=本地; langfuse=已配 LANGFUSE_*)")

课程环境配置:
  API Key:   ✗ 未配置
  LLM:       ollama / qwen3:4b
  Embedding: sentence-transformers / paraphrase-multilingual-MiniLM-L12-v2


[LLM] ollama / qwen3:4b


正在加载 Embedding 模型: paraphrase-multilingual-MiniLM-L12-v2 (device=cpu)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✓ 模型加载完成! 维度: 384
[Embedding] sentence-transformers / paraphrase-multilingual-MiniLM-L12-v2 (dim=384)
OK Observability backend: mock  (mock=本地; langfuse=已配 LANGFUSE_*)


.\utils\embedding_backend.py:86: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.dimension = self.model.get_sentence_embedding_dimension()


---

## 企业知识助手的架构

```
                           ┌───────────────────┐
                           │   User Question    │
                           └─────────┬─────────┘
                                     ▼
                       ┌──────────────────────────┐
              with span("capstone")
                       │   PlannerAgent (Multi-Agent) │
                       │   决定走哪条路径           │
                       └──────┬─────────┬─────────┘
                              │         │
                ┌─────────────┘         └────────────┐
                ▼                                     ▼
       ┌─────────────────┐                ┌────────────────────┐
       │  Agentic RAG     │                │  MCP Tool Call    │
       │  (Hybrid + Self) │                │  (订单/库存/通知) │
       └────────┬─────────┘                └─────────┬─────────┘
                │                                     │
                └──────────┐         ┌────────────────┘
                           ▼         ▼
                   ┌────────────────────────────┐
                   │  ReviewerAgent             │
                   │  检查回答质量              │
                   └─────────┬──────────────────┘
                             ▼
                   ┌──────────────────────┐
                   │  Final Answer        │
                   └──────────────────────┘

           整个过程：Langfuse 全程 trace
```

**4 大组件来源**：
| 组件 | 来源 |
|---|---|
| Multi-Agent (Planner/Worker/Reviewer) | Day 4 上午 + `utils/multi_agent.py` |
| MCP Server (订单/库存/通知) | Day 4 下午 + `mcp_server_demo/server.py` |
| Agentic RAG (Hybrid + Self) | Day 5 上午（在此简化版用 vector + 简单 self-check） |
| Observability | Day 5 下午（本节）+ `utils/observability.py` |

讲师版下面**完整跑通一遍**给学员看；学员在 Exercise 3 自己改其中 1-2 个组件。


In [3]:
# ── Step 1: 建知识库 (复用 Day5 上午的 KNOWLEDGE_DOCS) ──
KNOWLEDGE_DOCS = [
    {"id": "hr_01", "text": "公司年假政策：入职 5 年以下每年 5 天，5-10 年 10 天，10 年以上 15 天。", "category": "hr"},
    {"id": "hr_02", "text": "病假需出示三甲医院证明，全薪连续不超过 30 天。", "category": "hr"},
    {"id": "tech_01", "text": "API 限流：免费版 60 req/min，企业版 6000 req/min。", "category": "tech"},
    {"id": "tech_02", "text": "API 鉴权使用 Bearer Token；token 由 CONSOLE 生成，30 天过期。", "category": "tech"},
    {"id": "tech_03", "text": "出现 429 (Too Many Requests) 时建议指数退避重试。", "category": "tech"},
    {"id": "prod_01", "text": "StarLink 基础版 199 元/月，5 路并发；企业版 1999 元/月，100 路并发。", "category": "product"},
    {"id": "prod_03", "text": "SKU-A100 是 StarLink 入门套件，包含 1 个网关 + 5 个传感器，售价 4999 元。", "category": "product"},
]

vector_store = SimpleVectorStore(embedding_backend=embedder)
vector_store.add_documents([d["text"] for d in KNOWLEDGE_DOCS], metadata=[{"id": d["id"], "category": d["category"]} for d in KNOWLEDGE_DOCS],
)
print(f"OK 知识库就位：{len(KNOWLEDGE_DOCS)} 条")


✓ 已添加 7 个文档，总计 7 个
OK 知识库就位：7 条


In [4]:
# ── Step 2: 建 MCP server (复用 Day4 下午的 mcp_server_demo) ──
from server import build_server as build_mcp_server
mcp_server = build_mcp_server()
mcp_client = EduMCPClient(user_id="capstone-user")
mcp_client.connect(mcp_server)
print(f"OK MCP server 就位：{len(mcp_client.list_all_tools())} 个 tool")


OK MCP server 就位：3 个 tool


In [5]:
# ── Step 3: 建 Multi-Agent (Planner / RAG_Worker / MCP_Worker / Reviewer) ──
PLANNER_CAPSTONE_PROMPT = '''你是 **路由 Planner**。
判断用户问题应走哪条路径，输出 JSON：
- 知识/文档/政策类 → {"path": "rag"}
- 订单/库存/通知 → {"path": "mcp"}
- 闲聊/无法判断 → {"path": "direct"}

只输出 JSON。'''

RAG_WORKER_PROMPT = '''你是 **RAG Worker**。基于检索到的文档简洁答用户问题。
如果文档不够，明确说『信息不足』。'''

MCP_WORKER_PROMPT = '''你是 **MCP Worker**。
用户问题: {query}
可用 MCP tools: {tools}
请输出 JSON: {{"tool": "...", "arguments": {{...}}}}。'''

REVIEWER_CAPSTONE_PROMPT = '''你是 **回答审查者**。
判断这个回答是否：(1) 正面回答问题  (2) 不含编造  (3) 长度适中
APPROVE 或 REJECT，附一句理由。'''

planner = BaseAgent("Planner", llm, PLANNER_CAPSTONE_PROMPT, temperature=0.0)
rag_worker = BaseAgent("RAGWorker", llm, RAG_WORKER_PROMPT, temperature=0.1)
reviewer = BaseAgent("Reviewer", llm, REVIEWER_CAPSTONE_PROMPT, temperature=0.0)

print("OK 4 Agent 就位 (Planner / RAGWorker / MCPWorker(动态) / Reviewer)")


OK 4 Agent 就位 (Planner / RAGWorker / MCPWorker(动态) / Reviewer)


In [6]:
# -- Step 4: production pipeline with trace --
# The point of this capstone is the before/after: a broad router fails on
# product-vs-inventory ambiguity; the production route adds deterministic guards.
import re


def _extract_json_object(raw):
    text = raw.strip()
    if text.startswith("```"):
        text = text.split("```")[1].lstrip("json").strip()
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError(f"No JSON object found in LLM output: {text[:120]}")
    return json.loads(m.group(0))


SUPPORTED_RAG_TERMS = [
    "年假", "病假", "限流", "鉴权", "Bearer", "token", "429",
    "StarLink", "基础版", "企业版", "SKU-A100", "API",
]
UNSUPPORTED_COMPANY_TERMS = ["医疗保险", "加班补贴", "团建", "报销", "福利", "公司有"]


def _is_supported_rag_query(query):
    return any(term.lower() in query.lower() for term in SUPPORTED_RAG_TERMS)


def _rule_route(query):
    q_upper = query.upper()
    q_lower = query.lower()
    if re.search(r"\bORD-[A-Z0-9]+\b", q_upper) or "订单" in query:
        return "mcp"
    if re.search(r"\bSKU-[A-Z0-9]+\b", q_upper) and any(term in query or term in q_lower for term in ["库存", "还有多少", "stock", "inventory"]):
        return "mcp"
    if any(term in query for term in ["通知", "提醒", "告知"]):
        return "mcp"
    if _is_supported_rag_query(query):
        return "rag"
    if any(term in query for term in UNSUPPORTED_COMPANY_TERMS):
        return "direct"
    return None


@observe("capstone.route_llm")
def route_query_llm(query):
    plan_msg = Message("user", "Planner", MessageType.TASK, query)
    raw = planner.receive(plan_msg).payload.strip()
    try:
        return _extract_json_object(raw).get("path", "direct")
    except Exception:
        return "direct"


@observe("capstone.route")
def route_query(query):
    """Production route: deterministic guards first, LLM planner second."""
    return _rule_route(query) or route_query_llm(query)


@observe("capstone.rag_branch")
def rag_branch(query):
    results = vector_store.search(query, top_k=3)
    if not results:
        return "知识库未覆盖这个问题，建议补充文档后再回答。"
    ctx = "\n".join(f"- {r['document']}" for r in results)
    rag_msg = Message(
        "Planner", "RAGWorker", MessageType.TASK,
        f"问题: {query}\n\n检索到:\n{ctx}",
    )
    answer = rag_worker.receive(rag_msg).payload.strip()
    sources = "\n".join(f"[{r['metadata'].get('id')}] {r['document']}" for r in results[:2])
    return f"{answer}\n\n依据文档:\n{sources}"


def _fallback_mcp_plan(query):
    q_upper = query.upper()
    order = re.search(r"\bORD-[A-Z0-9]+\b", q_upper)
    if order:
        return {"tool": "query_order", "arguments": {"order_id": order.group(0)}}
    sku = re.search(r"\bSKU-[A-Z0-9]+\b", q_upper)
    if sku:
        return {"tool": "check_inventory", "arguments": {"sku": sku.group(0)}}
    notify = re.search(r"(?:通知|提醒|告知)\s*([A-Za-z][\w-]*)", query)
    if notify:
        return {"tool": "send_notification", "arguments": {"user_id": notify.group(1), "message": query}}
    return None


def _normalize_mcp_plan(plan, query):
    normalized = {"tool": plan.get("tool"), "arguments": dict(plan.get("arguments") or {})}
    q_upper = query.upper()
    order = re.search(r"\bORD-[A-Z0-9]+\b", q_upper)
    if normalized["tool"] == "query_order" and order:
        normalized["arguments"]["order_id"] = order.group(0)
    sku = re.search(r"\bSKU-[A-Z0-9]+\b", q_upper)
    if normalized["tool"] == "check_inventory" and sku:
        normalized["arguments"]["sku"] = sku.group(0)
    return normalized


@observe("capstone.mcp_branch")
def mcp_branch(query):
    tools = mcp_client.list_all_tools()
    tools_desc = "; ".join(f"{t['name']}({list(t['parameters']['properties'].keys())})" for t in tools)
    raw = llm.generate(MCP_WORKER_PROMPT.format(query=query, tools=tools_desc), temperature=0.0).strip()
    planner_source = "llm"
    try:
        plan = _extract_json_object(raw)
        if not isinstance(plan.get("arguments"), dict):
            raise ValueError("工具参数应为 JSON 对象")
    except Exception:
        plan = _fallback_mcp_plan(query)
        planner_source = "rule-fallback"
    if plan is None:
        return f"[MCP 调用失败] 无法从问题中提取 tool 参数；LLM 输出: {raw[:120]}"
    try:
        result = mcp_client.call(mcp_server.name, plan["tool"], **plan["arguments"])
        return f"调用 MCP tool {plan['tool']}({plan['arguments']}) [{planner_source}] 得到: {result}"
    except Exception as e:
        return f"[MCP 调用失败] {e}"


@observe("capstone.direct_answer")
def direct_branch(query):
    if any(term in query for term in UNSUPPORTED_COMPANY_TERMS):
        return "当前知识库未覆盖这个问题，不能确认是否有相关政策；建议联系 HR 或系统管理员补充文档后再答复。"
    return llm.generate(f"用一两句话简短回答: {query}", temperature=0.3).strip()


@observe("capstone.review")
def review_answer(query, answer):
    rev_msg = Message("user", "Reviewer", MessageType.REVIEW, f"问题: {query}\n\n回答: {answer}")
    return reviewer.receive(rev_msg).payload


@observe("capstone.baseline_rag_pipeline")
def baseline_rag_pipeline(query):
    answer = rag_branch(query)
    return {"path": "rag-only", "answer": answer, "review": ""}


@observe("capstone.naive_keyword_pipeline")
def naive_keyword_pipeline(query):
    """Before version: routes broad product/SKU keywords to MCP too eagerly."""
    q_upper = query.upper()
    if "ORD-" in q_upper or "SKU-" in q_upper or "STARLINK" in q_upper or "订单" in query or "库存" in query:
        path = "mcp"
    elif _is_supported_rag_query(query):
        path = "rag"
    else:
        path = "direct"
    if path == "rag":
        answer = rag_branch(query)
    elif path == "mcp":
        answer = mcp_branch(query)
    else:
        answer = direct_branch(query)
    review = review_answer(query, answer)
    return {"path": path, "answer": answer, "review": review}


@observe("capstone.llm_planner_pipeline")
def llm_planner_pipeline(query):
    path = route_query_llm(query)
    if path == "rag":
        answer = rag_branch(query)
    elif path == "mcp":
        answer = mcp_branch(query)
    else:
        answer = direct_branch(query)
    review = review_answer(query, answer)
    return {"path": path, "answer": answer, "review": review}


@observe("capstone.production_pipeline")
def production_pipeline(query):
    path = route_query(query)
    if path == "rag":
        answer = rag_branch(query)
    elif path == "mcp":
        answer = mcp_branch(query)
    else:
        answer = direct_branch(query)
    review = review_answer(query, answer)
    return {"path": path, "answer": answer, "review": review}


@observe("capstone.upgraded_pipeline")
def upgraded_pipeline(query):
    """Public capstone entrypoint: production version."""
    return production_pipeline(query)


observer.reset()
for q in [
    "入职 8 年有几天年假？",
    "查一下订单 ORD-002",
    "SKU-A100 是什么？多少钱？",
    "你好",
]:
    print("=" * 60)
    print(f"Q: {q}")
    r = upgraded_pipeline(q)
    print(f"  路径: {r['path']}")
    print(f"  答: {r['answer'][:180]}")
    print(f"  Review: {r['review'][:80]}")

print("\n" + "=" * 60)
print("Trace tree:")
observer.print_tree()
print("\nSummary:")
print(json.dumps(observer.summary(), ensure_ascii=False, indent=2))

Q: 入职 8 年有几天年假？


  路径: rag
  答: 入职8年有10天年假。

依据文档:
[hr_01] 公司年假政策：入职 5 年以下每年 5 天，5-10 年 10 天，10 年以上 15 天。
[hr_02] 病假需出示三甲医院证明，全薪连续不超过 30 天。
  Review: APPROVE  
理由：回答直接基于文档[hr_01]准确说明入职8年对应10天年假，正面回答问题、无编造且长度简洁适中。
Q: 查一下订单 ORD-002


  路径: mcp
  答: 调用 MCP tool query_order({'order_id': 'ORD-002'}) [llm] 得到: {"status": "pending", "total": 89.0, "customer": "bob"}
  Review: APPROVE，因为回答直接提供了订单查询结果（状态pending、总金额89.0、客户bob），正面回应了问题，信息具体且无虚构内容，同时长度简洁适中。
Q: SKU-A100 是什么？多少钱？


  路径: rag
  答: SKU-A100 是 StarLink 入门套件，包含 1 个网关和 5 个传感器，售价 4999 元。

依据文档:
[prod_03] SKU-A100 是 StarLink 入门套件，包含 1 个网关 + 5 个传感器，售价 4999 元。
[prod_01] StarLink 基础版 199 元/月，5 路并发；企业版 1999 元/月，100 路并
  Review: 
Q: 你好


  路径: direct
  答: 你好！有什么需要帮忙的吗？
  Review: APPROVE  
因为回答直接正面回应了“你好”问题，不含任何编造内容，且长度简洁适中（仅5个中文字符）。

Trace tree:
├─ capstone.upgraded_pipeline (7792ms)
  ├─ capstone.production_pipeline (7792ms)
    ├─ capstone.route (1ms)
    ├─ capstone.rag_branch (4155ms)
    ├─ capstone.review (3635ms)
├─ capstone.upgraded_pipeline (11841ms)
  ├─ capstone.production_pipeline (11841ms)
    ├─ capstone.route (0ms)
    ├─ capstone.mcp_branch (2525ms)
    ├─ capstone.review (9316ms)
├─ capstone.upgraded_pipeline (16631ms)
  ├─ capstone.production_pipeline (16631ms)
    ├─ capstone.route (0ms)
    ├─ capstone.rag_branch (4739ms)
    ├─ capstone.review (11893ms)
├─ capstone.upgraded_pipeline (7717ms)
  ├─ capstone.production_pipeline (7717ms)
    ├─ capstone.route (1343ms)
      ├─ capstone.route_llm (1343ms)
    ├─ capstone.direct_answer (3109ms)
    ├─ capstone.review (3266ms)

Summary:
{
  "n_traces": 4,
  "n_total_spans": 21,
  "total_duration_ms": 43981.5,
  "tokens": {
    "prompt": 204,
    "completi

In [7]:
# [LEARNER_FILL] 难度=基础+进阶 | 提示=约8-14行
# ============================================================
# Checkpoint 1 | 全栈 Capstone 跑通 + trace 验证
# ============================================================
#
# 【基础】（人人必做，10 min）
#   实现 capstone_quick_test()：跑 3 个 query (rag/mcp/direct 各一)，验证：
#   - 每个 query 都返回 {path, answer, review}
#   - observer 至少记录了 3 个根 span
#
# 【进阶】（技术学员选做，15 min）
#   实现 capstone_with_fallback(query)：
#   - 走 rag → 如果 review 含 'REJECT' 或 answer 含 'Fallback' → 退化到 direct
#   - 走 mcp → 如果调用失败 → 退化到 direct
#   - 返回 {path, fallback_used, answer}
# ============================================================

def capstone_quick_test():
    """【基础】跑 3 个 query 验证 pipeline + trace 工作"""
    # ↓↓↓ 【基础】填空（约 8 行）↓↓↓
    observer.reset()
    queries = ["StarLink 基础版价格", "查订单 ORD-001", "你好"]
    results = []
    for q in queries:
        r = upgraded_pipeline(q)
        results.append(r)
    return {"results": results, "n_traces": len(observer.spans), "summary": observer.summary()}
    # ↑↑↑ 【基础】结束 ↑↑↑


def capstone_with_fallback(query):
    """【进阶】RAG/MCP 失败时降级 direct"""
    # ↓↓↓ 【进阶】填空（约 14 行）↓↓↓
    path = route_query(query)
    fallback_used = False
    try:
        if path == "rag":
            answer = rag_branch(query)
            if "Fallback" in answer or "信息不足" in answer:
                answer = direct_branch(query)
                fallback_used = True
                path = "rag→direct"
        elif path == "mcp":
            answer = mcp_branch(query)
            if "失败" in answer:
                answer = direct_branch(query)
                fallback_used = True
                path = "mcp→direct"
        else:
            answer = direct_branch(query)
    except Exception as e:
        answer = direct_branch(query)
        fallback_used = True
        path = f"{path}→direct (error: {e})"
    return {"path": path, "fallback_used": fallback_used, "answer": answer}
    # ↑↑↑ 【进阶】结束 ↑↑↑


def verify():
    print("=" * 56); print("【基础】capstone_quick_test"); print("=" * 56)
    try:
        result = capstone_quick_test()
        for r in result["results"]:
            assert "path" in r and "answer" in r and "review" in r
        print(f"  3 query 全部跑通")
        print(f"  根 span 数: {result['n_traces']}")
        print(f"  Summary: {result['summary']}")
        assert result["n_traces"] >= 3
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】capstone_with_fallback"); print("=" * 56)
    try:
        # 触发 OOD → fallback
        r1 = capstone_with_fallback("公司有没有团建活动？")
        print(f"  '公司有团建吗?' → path={r1['path']}, fallback={r1['fallback_used']}")
        # 正常 RAG
        r2 = capstone_with_fallback("入职 5 年年假几天？")
        print(f"  '入职 5 年年假?' → path={r2['path']}, fallback={r2['fallback_used']}")
        assert "path" in r1 and "answer" in r1
        print("OK 进阶通过")
    except NotImplementedError:
        print("SKIP 进阶跳过（未实现）")
    except Exception as e:
        print(f"FAIL 进阶未通过: {type(e).__name__}: {e}")

verify()

【基础】capstone_quick_test


  3 query 全部跑通
  根 span 数: 3
  Summary: {'n_traces': 3, 'n_total_spans': 16, 'total_duration_ms': 26263.1, 'tokens': {'prompt': 131, 'completion': 559, 'total': 690}, 'cost_usd': 0.0007}
OK 基础通过

【进阶】capstone_with_fallback
  '公司有团建吗?' → path=direct, fallback=False


  '入职 5 年年假?' → path=rag, fallback=False
OK 进阶通过


In [8]:
# [LEARNER_FILL] Difficulty: basic + advanced
# ============================================================
# Checkpoint 2 | batch eval: before/after production comparison
# ============================================================
import random
from Applications.skills_demo.capstone_assistant.answer_checks import matches_facts


def batch_eval(pipeline_fn, dataset_path):
    items = [json.loads(line) for line in open(dataset_path, encoding="utf-8")]
    results = []
    cat_correct = {}
    cat_total = {}
    for it in items:
        out = pipeline_fn(it["query"])
        answer = out.get("answer", "") if isinstance(out, dict) else str(out)
        ok = matches_facts(answer, it)
        cat = it["category"]
        cat_total[cat] = cat_total.get(cat, 0) + 1
        if ok:
            cat_correct[cat] = cat_correct.get(cat, 0) + 1
        results.append({
            "query": it["query"],
            "answer": answer[:140],
            "ok": ok,
            "cat": cat,
            "path": out.get("path") if isinstance(out, dict) else None,
        })
    success = sum(1 for r in results if r["ok"])
    per_cat = {c: cat_correct.get(c, 0) / cat_total[c] for c in cat_total}
    return {"success_rate": success / len(items), "per_category": per_cat, "details": results}


def compare_pipelines(dataset_path="data/eval_dataset.jsonl"):
    return {
        "Pure RAG baseline": batch_eval(baseline_rag_pipeline, dataset_path),
        "Before: broad keyword router": batch_eval(naive_keyword_pipeline, dataset_path),
        "After: production guards": batch_eval(production_pipeline, dataset_path),
    }


def _print_eval_table(comparison):
    for name, result in comparison.items():
        print(f"  {name:<30} {result['success_rate']:.0%}")
    before = comparison["Before: broad keyword router"]["success_rate"]
    after = comparison["After: production guards"]["success_rate"]
    print(f"\n  与 Before 的要点命中率差: {(after - before) * 100:+.1f} percentage points")


def verify():
    print("=" * 56); print("【基础】batch_eval 前后对比"); print("=" * 56)
    try:
        global last_eval_comparison
        last_eval_comparison = compare_pipelines("data/eval_dataset.jsonl")
        _print_eval_table(last_eval_comparison)
        print("\n  After 分类:")
        for cat, acc in last_eval_comparison["After: production guards"]["per_category"].items():
            print(f"    {cat:<10} {acc:.0%}")
        print("\n  Before 失败 case (前 5):")
        fails = [r for r in last_eval_comparison["Before: broad keyword router"]["details"] if not r["ok"]][:5]
        for f in fails:
            print(f"    [{f['cat']}/{f['path']}] {f['query']}")
            print(f"      A: {f['answer'][:100]}")
        assert "After: production guards" in last_eval_comparison
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return


verify()

【基础】batch_eval 前后对比


  Pure RAG baseline              77%
  Before: broad keyword router   69%
  After: production guards       92%

  与 Before 的要点命中率差: +23.1 percentage points

  After 分类:
    hr         100%
    tech       100%
    product    100%
    mcp        50%
    ood        100%
    direct     100%

  Before 失败 case (前 5):
    [product/mcp] StarLink 基础版多少钱？
      A: 调用 MCP tool check_inventory({'sku': 'starlink_basic'}) [llm] 得到: {"error": "SKU starlink_basic not i
    [product/mcp] StarLink 企业版价格 + 并发是多少？
      A: 调用 MCP tool check_inventory({'sku': 'starlink-enterprise'}) [llm] 得到: {"error": "SKU starlink-enterp
    [product/mcp] SKU-A100 是什么？多少钱？
      A: 调用 MCP tool check_inventory({'sku': 'SKU-A100'}) [rule-fallback] 得到: {"sku": "SKU-A100", "quantity":
    [mcp/mcp] SKU-A100 库存还有多少？
      A: [MCP 调用失败] Tool 'check_inventory' arg 'sku' expected string, got list (['SKU-A100'])
OK 基础通过



In [9]:
policy = next(doc["text"] for doc in KNOWLEDGE_DOCS if doc["id"] == "tech_01")
answers = {
    "A": "免费版每分钟 60 次，企业版每分钟 6000 次。",
    "B": "免费版每分钟 60 次，企业版每秒 6000 次。",
    "C": "企业版每分钟 6000 次。",
}
print(policy)
print("问题：免费版和企业版各允许多少次 API 请求？")
for key, answer in answers.items():
    print(f"{key}. {answer}")

# 对照原文给每条回答打分：每个版本的次数和时间单位都正确，得 1 分，满分 2 分。
# 把 None 换成你的分数，再运行下一格。
my_scores = {"A": None, "B": None, "C": None}

API 限流：免费版 60 req/min，企业版 6000 req/min。
问题：免费版和企业版各允许多少次 API 请求？
A. 免费版每分钟 60 次，企业版每分钟 6000 次。
B. 免费版每分钟 60 次，企业版每秒 6000 次。
C. 企业版每分钟 6000 次。


In [10]:
reference_scores = {
    "A": (2, "两个版本的次数和单位都符合原文。"),
    "B": (1, "企业版写成了每秒，原文是每分钟。"),
    "C": (1, "企业版正确，漏了免费版。"),
}

for key, answer in answers.items():
    prompt = (
        f"政策原文：{policy}\n问题：免费版和企业版各允许多少次 API 请求？\n回答：{answer}\n"
        "只根据政策评分。免费版和企业版各占 1 分，每个版本的次数和时间单位都答对才得分，"
        "缺少或答错得 0 分。只返回总分，写一个数字 0、1 或 2。"
    )
    reply = llm.chat([{"role": "user", "content": prompt}], temperature=0).strip()
    model_score = reply if reply in {"0", "1", "2"} else f"未给出单个分数（原文 {reply[:80]!r}）"
    expected, reason = reference_scores[key]
    student = my_scores[key] if my_scores[key] is not None else "未填"
    print(f"{key}. 你的评分 {student}，模型 {model_score}，按原文 {expected}。{reason}")

A. 你的评分 未填，模型 2，按原文 2。两个版本的次数和单位都符合原文。


B. 你的评分 未填，模型 1，按原文 1。企业版写成了每秒，原文是每分钟。


C. 你的评分 未填，模型 1，按原文 1。企业版正确，漏了免费版。


In [11]:
# [LEARNER_FILL] Difficulty: basic + advanced
# ============================================================
# Checkpoint 3 | system reflection from real eval failures
# ============================================================

def reflect_on_capstone():
    comparison = globals().get("last_eval_comparison") or compare_pipelines("data/eval_dataset.jsonl")
    before = comparison["Before: broad keyword router"]
    after = comparison["After: production guards"]
    failures = [r for r in after["details"] if not r["ok"]][:3]
    if failures:
        suggestion = "建议：把剩余失败 case 回填到知识库 / MCP 工具 / 路由规则，不要只改 prompt。"
    else:
        suggestion = "建议：下一步接真实业务日志扩充 eval，特别是产品/SKU 和订单工具边界 case。"
    return {
        "before_rate": before["success_rate"],
        "after_rate": after["success_rate"],
        "top_3_failures": failures,
        "improvement_suggestion": suggestion,
    }


def trace_top_slow_paths(observer_obj, n=3):
    all_spans = []
    def collect(span_obj, depth=0):
        if span_obj.duration_ms is not None:
            all_spans.append({"name": span_obj.name, "depth": depth, "ms": span_obj.duration_ms})
        for child in span_obj.children:
            collect(child, depth + 1)
    for root in observer_obj.spans:
        collect(root)
    all_spans.sort(key=lambda x: -x["ms"])
    top = all_spans[:n]
    suggestions = []
    for s in top:
        name = s["name"].lower()
        if "rag" in name or "retrieve" in name:
            tip = "考虑：缓存常见 query 的检索结果"
        elif "llm" in name or "generate" in name or "review" in name:
            tip = "考虑：用更小模型 / 减少不必要的 reviewer 调用"
        elif "mcp" in name:
            tip = "考虑：MCP 调用并发化或加超时"
        else:
            tip = "考虑：分析此 span 占比是否合理"
        suggestions.append({"span": s["name"], "ms": s["ms"], "tip": tip})
    return suggestions


def verify():
    print("=" * 56); print("【基础】reflect_on_capstone"); print("=" * 56)
    try:
        report = reflect_on_capstone()
        print(f"  Before 准确率: {report['before_rate']:.0%}")
        print(f"  After  准确率: {report['after_rate']:.0%}")
        print(f"  提升: {(report['after_rate'] - report['before_rate']) * 100:+.1f} percentage points")
        print("  After Top 3 失败:")
        if report["top_3_failures"]:
            for f in report["top_3_failures"]:
                print(f"    - {f['query']}")
        else:
            print("    - 无（该教学 eval 集全部通过）")
        print(f"\n  改造建议: {report['improvement_suggestion']}")
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】trace_top_slow_paths"); print("=" * 56)
    try:
        observer.reset()
        for q in ["StarLink 基础版多少钱？", "查订单 ORD-001"]:
            production_pipeline(q)
        slow = trace_top_slow_paths(observer, n=3)
        print("  最慢的 3 个 span:")
        for s in slow:
            print(f"    {s['ms']:>7.1f}ms  {s['span']}")
            print(f"             -> {s['tip']}")
        assert len(slow) >= 1
        print("OK 进阶通过")
    except NotImplementedError:
        print("SKIP 进阶跳过")
    except Exception as e:
        print(f"FAIL 进阶未通过: {type(e).__name__}: {e}")

verify()


【基础】reflect_on_capstone
  Before 准确率: 69%
  After  准确率: 92%
  提升: +23.1 percentage points
  After Top 3 失败:
    - SKU-A100 库存还有多少？

  改造建议: 建议：把剩余失败 case 回填到知识库 / MCP 工具 / 路由规则，不要只改 prompt。
OK 基础通过

【进阶】trace_top_slow_paths


  最慢的 3 个 span:
    14730.7ms  capstone.production_pipeline
             -> 考虑：分析此 span 占比是否合理
    10307.8ms  capstone.production_pipeline
             -> 考虑：分析此 span 占比是否合理
     7869.3ms  capstone.rag_branch
             -> 考虑：缓存常见 query 的检索结果
OK 进阶通过


---

## Capstone → Skill 打包

我们已经把 5 天所有内容合体跑出一个升级 Capstone。最后 30 min**演示如何把它打包成一个 Skill**，让其他团队（或你自己未来的项目）可以复用。

### Capstone Skill 文件夹

`skills_demo/capstone_assistant/` 已经打包好了：

```
capstone_assistant/
├── SKILL.md                # 何时调 + workflow（决定 Claude 选不选它）
├── pipeline.py             # upgraded_pipeline() 主入口（从本 notebook 抽出）
├── eval.py                 # batch_eval 复用
└── reference/
    ├── architecture.md     # 4 大组件 deep dive (按需载入)
    └── eval_cases.jsonl    # 10 个评测用例
```

### 演示：discover + route + invoke


In [12]:
# Step 1: discover skills
import sys
from pathlib import Path
sys.path.insert(0, "utils")
from utils.skills_helpers import discover_skills, match_skill_for_query, load_skill_progressive, validate_skill

skills = discover_skills("Applications/skills_demo")
print(f"发现 {len(skills)} 个 skills:")
for s in skills:
    print(f"  • {s.name}: {s.description[:80]}...")
    print(f"      version={s.version}, helpers={[p.name for p in s.helper_files]}")


发现 3 个 skills:
  • enterprise-knowledge-assistant: Company-internal assistant for HR policy, product pricing, technical API docs, a...
      version=1.0, helpers=['answer_checks.py', 'eval.py', 'pipeline.py']
  • code-review: Review Python code or a pull request for correctness, maintainability, tests, an...
      version=0.2, helpers=['helper.py']
  • db-query: Route order status, inventory, SKU, and customer notification requests to the en...
      version=0.1, helpers=[]


In [13]:
# Step 2: 用户来一个企业相关 query → LLM 路由到 capstone-assistant skill
test_queries = [
    "入职 7 年有几天年假？",
    "查 ORD-002 订单状态",
    "code review 这个函数",
]

for q in test_queries:
    picked = match_skill_for_query(q, skills, llm)
    print(f"  query: {q[:30]}")
    print(f"    → routed to: {picked.name if picked else '(none)'}")


  query: 入职 7 年有几天年假？
    → routed to: enterprise-knowledge-assistant


  query: 查 ORD-002 订单状态
    → routed to: db-query


  query: code review 这个函数
    → routed to: code-review


In [14]:
# Step 3: 加载并调用 capstone_assistant.pipeline.upgraded_pipeline
# (注意：pipeline.py 已自动 setup env + 建 vector store + 起 mcp server)

import importlib.util
spec = importlib.util.spec_from_file_location(
    "capstone_pipeline",
    "Applications/skills_demo/capstone_assistant/pipeline.py",
)
capstone_module = importlib.util.module_from_spec(spec)

print("加载 capstone_assistant skill 的 pipeline.py...")
print("(首次会 setup env + 建 vector store + 启 mcp server，需 5-10s)")
spec.loader.exec_module(capstone_module)

# 直接调用打包后的 pipeline
result = capstone_module.upgraded_pipeline("入职 8 年有几天年假？")
print(f"\n打包后 Capstone 调用结果:")
print(f"  path: {result['path']}")
print(f"  answer: {result['answer'][:120]}")
print(f"  review: {result['review'][:80]}")

加载 capstone_assistant skill 的 pipeline.py...
(首次会 setup env + 建 vector store + 启 mcp server，需 5-10s)
课程环境配置:
  API Key:   ✗ 未配置
  LLM:       ollama / qwen3:4b
  Embedding: sentence-transformers / paraphrase-multilingual-MiniLM-L12-v2
[LLM] ollama / qwen3:4b
正在加载 Embedding 模型: paraphrase-multilingual-MiniLM-L12-v2 (device=cpu)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

.\utils\embedding_backend.py:86: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.dimension = self.model.get_sentence_embedding_dimension()


✓ 模型加载完成! 维度: 384
[Embedding] sentence-transformers / paraphrase-multilingual-MiniLM-L12-v2 (dim=384)
✓ 已添加 7 个文档，总计 7 个



打包后 Capstone 调用结果:
  path: rag
  answer: 根据检索到的公司年假政策（入职 5 年以下每年 5 天，5-10 年每年 10 天），入职 8 年属于“5-10 年”范围，因此有 **10 天年假**。  

（注：检索到的文档中其他内容如病假政策和 429 错误处理与年假问题无关，故未
  review: APPROVE  
回答准确基于给定文档 [hr_01] 的年假政策（入职 5-10 年每年 10 天），直接计算出入职 8 年的年假天数为 10 天，且长度简


In [15]:
# [LEARNER_FILL] Difficulty: basic + advanced
# ============================================================
# Checkpoint 4 | Skill packaging sanity check
# ============================================================

def capstone_skill_health_check():
    skill_dir = "Applications/skills_demo/capstone_assistant" if Path("Applications/skills_demo/capstone_assistant").exists() else "skills_demo/capstone_assistant"
    val = validate_skill(skill_dir)
    fm, body = parse_skill_md(f"{skill_dir}/SKILL.md")
    return {
        "skill_dir": skill_dir,
        "validate": val,
        "name": fm.get("name"),
        "description_len": len(fm.get("description", "")),
        "body_len": len(body),
        "has_pipeline": Path(f"{skill_dir}/pipeline.py").exists(),
    }


def audit_capstone_description_boundary():
    """Do not force an uplift claim; verify the description keeps clean boundaries."""
    from dataclasses import replace
    from utils.skills_helpers import discover_skills, match_skill_for_query

    skills_root = "Applications/skills_demo" if Path("Applications/skills_demo").exists() else "skills_demo"
    skills = discover_skills(skills_root)
    improved_desc = (
        "Company-internal assistant for HR policy, product pricing, technical API docs, "
        "and order/inventory/notification workflows. Use only for internal company questions; "
        "prefer db-query for direct ORD/SKU inventory lookups and code-review for code."
    )
    cases = [
        ("入职 5 年年假几天", "enterprise-knowledge-assistant"),
        ("API 限流怎么算", "enterprise-knowledge-assistant"),
        ("StarLink 企业版多少钱", "enterprise-knowledge-assistant"),
        ("查 ORD-XXX 订单", "db-query"),
        ("review my python code", "code-review"),
        ("发个通知给 alice", "db-query"),
    ]

    def with_desc(desc):
        return [replace(s, description=desc) if s.name == "enterprise-knowledge-assistant" else s for s in skills]

    current_desc = next(s.description for s in skills if s.name == "enterprise-knowledge-assistant")

    def audit(skill_list):
        log = []
        correct = 0
        for q, expected in cases:
            picked = match_skill_for_query(q, skill_list, llm)
            picked_name = picked.name if picked else None
            ok = picked_name == expected
            log.append({"q": q, "expected": expected, "picked": picked_name, "ok": ok})
            correct += int(ok)
        return correct / len(cases), log

    before_acc, before_log = audit(with_desc(current_desc))
    after_acc, after_log = audit(with_desc(improved_desc))
    return {
        "before_accuracy": before_acc,
        "after_accuracy": after_acc,
        "before_log": before_log,
        "after_log": after_log,
        "new_description": improved_desc,
    }


def verify():
    print("=" * 56); print("【基础】capstone_skill_health_check"); print("=" * 56)
    try:
        health = capstone_skill_health_check()
        print(f"  skill_dir: {health['skill_dir']}")
        print(f"  validate: ok={health['validate']['ok']}")
        print(f"  name: {health['name']}")
        print(f"  desc 长度: {health['description_len']} 字符")
        print(f"  body 长度: {health['body_len']} 字符")
        print(f"  has pipeline.py: {health['has_pipeline']}")
        assert health["validate"]["ok"]
        assert health["name"] == "enterprise-knowledge-assistant"
        assert health["has_pipeline"]
        print("OK 基础通过\n")
    except NotImplementedError:
        print("SKIP 基础未实现\n"); return
    except Exception as e:
        print(f"FAIL 基础未通过: {type(e).__name__}: {e}\n"); return

    print("=" * 56); print("【进阶】description boundary audit"); print("=" * 56)
    try:
        result = audit_capstone_description_boundary()
        print(f"  Current description accuracy: {result['before_accuracy']:.0%}")
        print(f"  Candidate description accuracy: {result['after_accuracy']:.0%}")
        print("\n  边界 case:")
        for b, a in zip(result["before_log"], result["after_log"]):
            change = ""
            if b["picked"] != a["picked"]:
                change = f" (route {b['picked']} -> {a['picked']})"
            print(f"    {b['q'][:30]}: {'OK' if b['ok'] else 'NO'} -> {'OK' if a['ok'] else 'NO'}{change}")
        print("OK 进阶通过")
    except NotImplementedError:
        print("SKIP 进阶跳过")
    except Exception as e:
        print(f"FAIL 进阶未通过: {type(e).__name__}: {e}")

from utils.skills_helpers import parse_skill_md

verify()

【基础】capstone_skill_health_check
  skill_dir: Applications/skills_demo/capstone_assistant
  validate: ok=True
  name: enterprise-knowledge-assistant
  desc 长度: 240 字符
  body 长度: 1759 字符
  has pipeline.py: True
OK 基础通过

【进阶】description boundary audit


  Current description accuracy: 100%
  Candidate description accuracy: 100%

  边界 case:
    入职 5 年年假几天: OK -> OK
    API 限流怎么算: OK -> OK
    StarLink 企业版多少钱: OK -> OK
    查 ORD-XXX 订单: OK -> OK
    review my python code: OK -> OK
    发个通知给 alice: OK -> OK
OK 进阶通过


## 组件与对应章节

| 组件 | App 入口 |
|---|---|
| Multi-Agent (Planner/Worker/Reviewer) | App4 |
| MCP 工具 (订单/库存) | App5 |
| RAG（向量检索） | App2 |
| LLMOps Trace | App7 |
| Skill 打包 | App6 |

用于实际业务前，还要补上权限控制、业务评测集、灰度发布、告警和人工复核。

把这套 fork 成你公司业务的下一步：

1. 改 `pipeline.py` 里的路由逻辑，把 4 类 query 换成你业务的 query 类型
2. 改 MCP server 暴露你公司的真实 API（订单、库存、CRM、ERP）
3. 改 RAG 知识库换成你公司的内部文档
4. 改 `eval_cases.jsonl` 写你业务关心的评测用例
5. 接 Langfuse / 自家观测平台拿到真 dashboard

从评测结果中挑一条答错的问题，沿调用记录找出最早出错的一步。
